# 面试问题：Semantic Entropy 怎样衡量答案不确定性，什么时候应该弃答？

可以直接复述的回答是：第一，对同一问题采样多条回答。第二，不能把文字不同直接当作语义不同，应先按可验证含义聚类。第三，对语义簇概率计算熵，簇越分散表示模型对结论越不确定。第四，高风险问题需要更低弃答阈值，并结合事实检索或人工转接。第五，聚类器必须理解否定和数值，避免把相反结论合并。第六，要在同一批问题上比较字符串熵、语义熵、覆盖率和错误回答率。下面用医院预约 FAQ 的离线采样演示。

## 真实案例：医院预约助手决定回答还是转人工

五个脱敏问题分别涉及门诊时间、空腹要求、证件、儿童发热和抗生素剂量。每题保存 5 条模型采样文本，答案是教学构造，不构成医疗建议。系统目标不是诊断，而是在结论分歧或缺少患者信息时弃答并转人工。

In [1]:
questions = [  # 定义五个具有不同风险和采样分歧的问题
    {"id": "MED-01", "question": "周一门诊几点开始？", "risk": "low", "samples": ["门诊早上八点开始。", "开放时间是 8:00。", "周一八点可以挂号。", "上午 8 点开诊。", "九点开始。"]},  # 四条八点与一条九点形成小分歧
    {"id": "MED-02", "question": "抽血前需要空腹多久？", "risk": "medium", "samples": ["通常空腹八小时。", "建议至少 8 小时不进食。", "需要十二小时空腹。", "一般空腹 12 小时。", "八小时即可。"]},  # 八小时与十二小时结论接近对半
    {"id": "MED-03", "question": "首次就诊需要带什么？", "risk": "low", "samples": ["请带身份证和医保卡。", "携带医保卡与身份证。", "需要有效证件和医保卡。", "身份证、医保卡都要带。", "带身份证以及医保卡。"]},  # 五条表达不同但语义一致
    {"id": "MED-04", "question": "三岁儿童发热 40 度是否立即急诊？", "risk": "high", "samples": ["应立即联系急诊。", "建议马上急诊评估。", "需要尽快去急诊。", "可以先在家观察。", "暂时观察即可。"]},  # 高风险问题存在相反建议
    {"id": "MED-05", "question": "抗生素一次吃几片？", "risk": "high", "samples": ["需要药名和规格才能判断。", "无法在缺少处方时给剂量。", "一次一片。", "一次两片。", "请咨询开药医生。"]},  # 缺少药物规格且答案高度分散
]  # 结束五个 FAQ 场景
print("问题输入：id | risk | question | sample_count")  # 展示不确定性系统接收的真实字段
for item in questions:  # 逐条输出五个问题
    print(f"{item['id']} | {item['risk']:6} | {item['question']} | {len(item['samples'])}")  # 呈现风险等级和采样数量
print("MED-02 五条采样：")  # 展示一个明显分歧问题的原始回答
for sample in questions[1]["samples"]:  # 逐条输出空腹时间采样
    print("-", sample)  # 保留数值差异供语义聚类观察


问题输入：id | risk | question | sample_count
MED-01 | low    | 周一门诊几点开始？ | 5
MED-02 | medium | 抽血前需要空腹多久？ | 5
MED-03 | low    | 首次就诊需要带什么？ | 5
MED-04 | high   | 三岁儿童发热 40 度是否立即急诊？ | 5
MED-05 | high   | 抗生素一次吃几片？ | 5
MED-02 五条采样：
- 通常空腹八小时。
- 建议至少 8 小时不进食。
- 需要十二小时空腹。
- 一般空腹 12 小时。
- 八小时即可。


## Baseline / 基线：把每个不同字符串当作独立答案

字符串熵无法识别“早上八点”和“8:00”是同一结论，因此即使五条语义一致，也会得到很高不确定性。

In [2]:
import math  # 使用自然对数计算离散熵
from collections import Counter  # 使用计数器统计字符串和语义簇频率
def entropy(labels):  # 计算任意离散标签序列的 Shannon entropy
    counts = Counter(labels)  # 统计每个标签出现次数
    total = sum(counts.values())  # 计算采样总数
    return -sum((count / total) * math.log(count / total) for count in counts.values())  # 汇总每个簇的负对数概率
baseline_rows = []  # 收集五题的字符串熵和弃答决定
baseline_threshold = 0.8  # 设置教学字符串熵阈值
print("字符串基线：id | unique_strings | entropy | decision")  # 输出原始文本粒度的不确定性
for item in questions:  # 对每题五条采样直接计算字符串熵
    string_entropy = entropy(item["samples"])  # 每个不同回答文本形成独立类别
    decision = "abstain" if string_entropy > baseline_threshold else "answer"  # 高于阈值时弃答
    baseline_rows.append((item["id"], string_entropy, decision))  # 保存基线结果供对照
    print(f"{item['id']} | {len(set(item['samples']))} | {string_entropy:.3f} | {decision}")  # 展示语义一致题也被错误弃答


字符串基线：id | unique_strings | entropy | decision
MED-01 | 5 | 1.609 | abstain
MED-02 | 5 | 1.609 | abstain
MED-03 | 5 | 1.609 | abstain
MED-04 | 5 | 1.609 | abstain
MED-05 | 5 | 1.609 | abstain


## 核心实现：否定和数值感知的语义簇

教学聚类器把回答映射为可验证结论标签，例如 `open_8`、`fast_12h`、`urgent`。无法归入已知合同的回答保持独立标签，避免被强行合并。

In [3]:
def semantic_label(question_id, text):  # 把自然语言回答映射为问题特定的可验证语义
    normalized = text.replace(" ", "").lower()  # 去除空格并统一 ASCII 大小写
    if question_id == "MED-01":  # 门诊时间按八点或九点聚类
        return "open_8" if "八" in normalized or "8:" in normalized or "8点" in normalized else "open_9"  # 保留关键时间差异
    if question_id == "MED-02":  # 空腹要求按八小时或十二小时聚类
        return "fast_12h" if "十二" in normalized or "12" in normalized else "fast_8h"  # 数值不同不能合并
    if question_id == "MED-03":  # 证件表达均指向身份证加医保卡
        return "id_and_insurance" if ("身份证" in normalized or "有效证件" in normalized) and "医保卡" in normalized else "other_docs"  # 验证两个必要实体
    if question_id == "MED-04":  # 高热建议必须区分急诊和观察
        return "observe" if "观察" in normalized else "urgent"  # 使用动作语义而非词面相似
    if "无法" in normalized or "需要药名" in normalized or "咨询" in normalized:  # 剂量问题中的安全拒答构成一类
        return "need_more_information"  # 标记缺少处方信息
    if "一片" in normalized:  # 保留具体一片剂量结论
        return "dose_one"  # 返回一片剂量标签
    if "两片" in normalized:  # 保留具体两片剂量结论
        return "dose_two"  # 返回两片剂量标签
    return "unknown"  # 未识别回答单独进入未知簇
semantic_thresholds = {"low": 0.55, "medium": 0.50, "high": 0.35}  # 风险越高允许的语义熵越低
semantic_rows = []  # 收集五题语义簇、熵和决策
print("语义熵：id | clusters | entropy | threshold | decision")  # 输出核心聚类中间量
for item in questions:  # 对五题重新按结论统计
    labels = [semantic_label(item["id"], sample) for sample in item["samples"]]  # 为五条采样生成语义标签
    semantic_entropy = entropy(labels)  # 计算语义簇概率熵
    threshold = semantic_thresholds[item["risk"]]  # 根据风险等级选择弃答阈值
    decision = "abstain" if semantic_entropy > threshold else "answer"  # 结论分散时转人工
    semantic_rows.append({"id": item["id"], "labels": labels, "entropy": semantic_entropy, "decision": decision})  # 保存完整证据
    print(f"{item['id']} | {dict(Counter(labels))} | {semantic_entropy:.3f} | {threshold:.2f} | {decision}")  # 展示文本聚合后的真实分歧


语义熵：id | clusters | entropy | threshold | decision
MED-01 | {'open_8': 4, 'open_9': 1} | 0.500 | 0.55 | answer
MED-02 | {'fast_8h': 3, 'fast_12h': 2} | 0.673 | 0.50 | abstain
MED-03 | {'id_and_insurance': 5} | -0.000 | 0.55 | answer
MED-04 | {'urgent': 3, 'observe': 2} | 0.673 | 0.35 | abstain
MED-05 | {'need_more_information': 3, 'dose_one': 1, 'dose_two': 1} | 0.950 | 0.35 | abstain


## 失败案例与修正：忽略否定会把相反答案合并

“需要空腹”和“不需要空腹”共享大量词面。只看关键词的聚类器会放进同一簇，错误降低熵；修正必须让否定进入语义合同。

In [4]:
negation_samples = ["检查前需要空腹。", "检查前不需要空腹。", "需要空腹八小时。", "无需空腹。", "建议保持空腹。"]  # 构造含肯定与否定的五条回答
naive_negation_labels = ["fasting" if "空腹" in text else "other" for text in negation_samples]  # 只看关键词会把相反结论全部合并
def negation_aware_label(text):  # 显式识别“不需要”和“无需”否定模式
    return "no_fasting" if "不需要" in text or "无需" in text else "fasting"  # 将否定建议分到独立簇
safe_negation_labels = [negation_aware_label(text) for text in negation_samples]  # 对五条反例执行否定感知聚类
naive_negation_entropy = entropy(naive_negation_labels)  # 计算错误聚类得到的虚假低熵
safe_negation_entropy = entropy(safe_negation_labels)  # 计算保留相反结论后的真实熵
print("否定修正前：", dict(Counter(naive_negation_labels)), "entropy=", round(naive_negation_entropy, 3))  # 展示关键词聚类的过度自信
print("否定修正后：", dict(Counter(safe_negation_labels)), "entropy=", round(safe_negation_entropy, 3))  # 展示相反答案被正确分离


否定修正前： {'fasting': 5} entropy= -0.0
否定修正后： {'fasting': 3, 'no_fasting': 2} entropy= 0.673


## 结果表：逐问题回答覆盖与弃答原因

In [5]:
expected_decisions = {"MED-01": "answer", "MED-02": "abstain", "MED-03": "answer", "MED-04": "abstain", "MED-05": "abstain"}  # 定义教学场景的人工安全决定
baseline_correct = 0  # 初始化字符串熵决策正确数
semantic_correct = 0  # 初始化语义熵决策正确数
print("id | expected | string_decision | semantic_decision | dominant_semantics")  # 输出逐问题对照
for baseline, semantic in zip(baseline_rows, semantic_rows):  # 对齐两种方法的五题结果
    expected = expected_decisions[semantic["id"]]  # 获取人工安全决策
    baseline_correct += int(baseline[2] == expected)  # 累加字符串基线正确数
    semantic_correct += int(semantic["decision"] == expected)  # 累加语义方法正确数
    dominant = Counter(semantic["labels"]).most_common(1)[0]  # 获取最高频结论及样本数
    print(f"{semantic['id']} | {expected} | {baseline[2]} | {semantic['decision']} | {dominant}")  # 展示回答或弃答的证据
baseline_accuracy = baseline_correct / len(questions)  # 计算字符串熵决策准确率
semantic_accuracy = semantic_correct / len(questions)  # 计算语义熵决策准确率
semantic_coverage = sum(row["decision"] == "answer" for row in semantic_rows) / len(semantic_rows)  # 计算系统实际回答覆盖率
print(f"决策准确率：string={baseline_accuracy:.1%}，semantic={semantic_accuracy:.1%}；回答覆盖率={semantic_coverage:.1%}")  # 输出安全与覆盖权衡


id | expected | string_decision | semantic_decision | dominant_semantics
MED-01 | answer | abstain | answer | ('open_8', 4)
MED-02 | abstain | abstain | abstain | ('fast_8h', 3)
MED-03 | answer | abstain | answer | ('id_and_insurance', 5)
MED-04 | abstain | abstain | abstain | ('urgent', 3)
MED-05 | abstain | abstain | abstain | ('need_more_information', 3)
决策准确率：string=60.0%，semantic=100.0%；回答覆盖率=40.0%


## 结果解读

MED-03 的五种措辞全部落入 `id_and_insurance`，语义熵为零，因此不再因文字多样性错误弃答。MED-02、MED-04 和 MED-05 存在数值、动作或剂量分歧，风险阈值促使系统转人工。否定反例说明聚类错误会让系统虚假自信，语义熵不能脱离聚类器质量。

## 生产边界

生产实现通常使用 NLI、语义等价模型或结构化答案验证器，并在独立校准集选择风险分层阈值。采样温度、样本数和模型版本都会改变熵；医疗场景还需检索权威指南、记录弃答原因和提供紧急渠道。本例不输出医疗结论，只演示不确定性门禁。

## 最小回归测试

In [6]:
assert len(questions) >= 5 and all(len(item["samples"]) >= 5 for item in questions)  # 保证每个真实问题具有足够采样
assert next(row for row in semantic_rows if row["id"] == "MED-03")["entropy"] == 0.0  # 保证五种等价证件回答被合并
assert next(row for row in semantic_rows if row["id"] == "MED-04")["decision"] == "abstain"  # 保证高热相反建议触发弃答
assert safe_negation_entropy > naive_negation_entropy  # 保证否定感知修正暴露真实不确定性
assert semantic_accuracy > baseline_accuracy  # 保证语义聚类在同一人工决策集上优于字符串熵
assert semantic_coverage == 0.4  # 保证系统只回答两条低分歧问题
